# Delete a hypothesis and watch the theorem break

*Adversarial Mathematics From First Principles*, [chapter 11](https://programmer.ie/books/math/11-chapter/) · [worked example](https://programmer.ie/examples/math/11-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

The quickest way to see what a hypothesis does is to delete it. This program
deletes the right angle from Pythagoras' theorem and the condition |v| < 1 from
γ = 1/√(1 − v²), then tests the four repairs the chapter discusses: narrowing
the hypotheses, correcting the conclusion, a vacuous hypothesis, and a patch
that only excludes the known counterexample.

## Run it

Standard library only; nothing to install.

In [1]:
"""What a theorem assumes: delete a hypothesis, then repair honestly (standard library only)."""
import math


def vec(p, q):
    return (q[0] - p[0], q[1] - p[1])


def dot(u, w):
    return u[0] * w[0] + u[1] * w[1]


def d2(p, q):
    return dot(vec(p, q), vec(p, q))


def residual(A, B, C):            # d2(B,C) - d2(A,B) - d2(A,C)
    return d2(B, C) - d2(A, B) - d2(A, C)


triangles = [((0, 0), (1, 0), (1, 1)), ((0, 0), (1, 0), (2, 1)), ((0, 0), (1, 0), (0, 1)),
             ((2, -1), (5, 3), (-1, 7)), ((0, 0), (3, 0), (0, 4))]

print("Deleting the right angle: residual = -2 (AB . AC) for every triangle")
for A, B, C in triangles:
    r, ab_ac = residual(A, B, C), dot(vec(A, B), vec(A, C))
    print(f"  A={A} B={B} C={C}: residual {r:>4}, -2 AB.AC = {-2 * ab_ac:>4}")
    assert r == -2 * ab_ac

print("\nFour repairs of 'd2(B,C) = d2(A,B) + d2(A,C) for all A, B, C':")
holds = lambda hyp, concl: all(concl(*t) for t in triangles if hyp(*t))
satisfiable = lambda hyp: any(hyp(*t) for t in triangles)
repairs = {
    "1. narrow: assume AB.AC = 0": (lambda A, B, C: dot(vec(A, B), vec(A, C)) == 0,
                                    lambda A, B, C: residual(A, B, C) == 0),
    "2. correct: add -2 AB.AC":     (lambda A, B, C: True,
                                    lambda A, B, C: d2(B, C) == d2(A, B) + d2(A, C) - 2 * dot(vec(A, B), vec(A, C))),
    "3. vacuous: assume 0 = 1":     (lambda A, B, C: 0 == 1, lambda A, B, C: residual(A, B, C) == 0),
    "4. patch: assume AB.AC != 1":  (lambda A, B, C: dot(vec(A, B), vec(A, C)) != 1,
                                    lambda A, B, C: residual(A, B, C) == 0),
}
for name, (hyp, concl) in repairs.items():
    print(f"  {name:30} holds on the test set: {holds(hyp, concl)!s:5}  hypothesis satisfiable: {satisfiable(hyp)}")

print("\nDeleting |v| < 1 from gamma = 1/sqrt(1 - v^2), with a proof library's total conventions:")
lean_sqrt = lambda x: math.sqrt(x) if x >= 0 else 0.0     # Mathlib: sqrt of a negative is 0
lean_div = lambda a, b: a / b if b != 0 else 0.0           # Mathlib: division by 0 is 0
for v in (0.6, 2.0):
    g = lean_div(1.0, lean_sqrt(1 - v * v))
    print(f"  v = {v}: gamma = {g}, gamma^2 (1 - v^2) = {g * g * (1 - v * v) + 0.0}  (should be 1)")

Deleting the right angle: residual = -2 (AB . AC) for every triangle
  A=(0, 0) B=(1, 0) C=(1, 1): residual   -2, -2 AB.AC =   -2
  A=(0, 0) B=(1, 0) C=(2, 1): residual   -4, -2 AB.AC =   -4
  A=(0, 0) B=(1, 0) C=(0, 1): residual    0, -2 AB.AC =    0
  A=(2, -1) B=(5, 3) C=(-1, 7): residual  -46, -2 AB.AC =  -46
  A=(0, 0) B=(3, 0) C=(0, 4): residual    0, -2 AB.AC =    0

Four repairs of 'd2(B,C) = d2(A,B) + d2(A,C) for all A, B, C':
  1. narrow: assume AB.AC = 0    holds on the test set: True   hypothesis satisfiable: True
  2. correct: add -2 AB.AC       holds on the test set: True   hypothesis satisfiable: True
  3. vacuous: assume 0 = 1       holds on the test set: True   hypothesis satisfiable: False
  4. patch: assume AB.AC != 1    holds on the test set: False  hypothesis satisfiable: True

Deleting |v| < 1 from gamma = 1/sqrt(1 - v^2), with a proof library's total conventions:
  v = 0.6: gamma = 1.25, gamma^2 (1 - v^2) = 1.0  (should be 1)
  v = 2.0: gamma = 0.0, gamma^2 (1 - 

## What the output shows

Without the right angle, the residual is always −2 AB·AC: the deleted
hypothesis is exactly what makes it zero. Repairs 1 and 2 are honest. Repair 3
"holds" only because its hypothesis can never be satisfied, so it says nothing.
Repair 4 fails on the second triangle: excluding known witnesses is not a
repair. Without |v| < 1, a proof library's total square root and division
quietly give γ = 0 at v = 2, so the conclusion fails with no error message.

## Not shown

- The repairs are tested on five triangles, not proved; repair 2 is the law of cosines, which needs its own proof.
- The two lambdas imitate Mathlib's conventions for √ and ÷ in floating point; they are not Lean.

## Challenge it

Examine the four repairs of "d²(B, C) = d²(A, B) + d²(A, C) for all A, B, C".
For each, decide whether it is true, whether its hypothesis is satisfiable,
and whether it says anything useful. Find a further triangle that refutes
repair 4. Then explain why deleting |v| < 1 from the gamma bridge fails
silently in a proof assistant rather than with an error.